In [1]:
#Environment Setup
# 1. Install the Stellar SDK (Necessary for every Colab session)
!pip install --upgrade stellar_sdk


from stellar_sdk import Server
import pandas as pd

# 1. Initialize Stellar Server (Horizon)
server = Server("https://horizon.stellar.org")

def detect_fraud(account_id, internal_limit=1000):
    # Fetch On-Chain operations for the account
    try:
        ops = server.operations().for_account(account_id).order(desc=True).limit(10).call()
        records = ops['_embedded']['records']

        fraud_alerts = []
        for op in records:
            # Look for DMMK to USDT swaps (path_payment or manage_sell_offer)
            if op['type'] in ['path_payment_strict_receive', 'manage_sell_offer']:
                amount = float(op.get('amount', 0) or op.get('starting_amount', 0))

                # Logic: If On-Chain swap > Internal authorized limit, it's a Bypass
                if amount > internal_limit:
                    fraud_alerts.append({
                        "tx_hash": op['transaction_hash'],
                        "type": "Bypass Detection",
                        "severity": "High"
                    })
        return fraud_alerts
    except Exception as e:
        return f"Error fetching data: {e}"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 771.0/771.0 kB 12.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 92.7/92.7 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 49.9 MB/s eta 0:00:00
